# W14C2 Lab: Making Your Project Reproducible

Run every cell from the top. **Everything already works.**

This is the lab that protects your project grade. Everything here applies
directly to the code you are about to submit.

Today you will:

1. Run the same code twice and get two different answers.
2. Fix it, and prove the fix worked.
3. Write the environment record your final report needs.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import platform
import random
import sys

import numpy as np
import torch

print("python  ", sys.version.split()[0])
print("platform", platform.system(), platform.machine())
print("numpy   ", np.__version__)
print("torch   ", torch.__version__)

## Part 1. The same code, two different answers

Anything that samples is nondeterministic by default. That includes model
initialisation, dropout, shuffling, and every generation call you make.

In [ ]:
# GIVEN. Run the same experiment twice, unseeded.
def experiment():
    weights = torch.randn(3)
    noise = np.random.randn(3)
    pick = random.choice(["a", "b", "c"])
    return weights.sum().item(), noise.sum(), pick

first = experiment()
second = experiment()
print("run 1:", tuple(round(x, 4) if isinstance(x, float) else x for x in first))
print("run 2:", tuple(round(x, 4) if isinstance(x, float) else x for x in second))
print()
print("identical:", first == second)
print("If your results table came from a run like this, nobody can check it.")

In [ ]:
# ================== YOUR TURN 1 ==================
# Seed all three sources of randomness and prove the runs match.
#
# You need torch.manual_seed, np.random.seed and random.seed. Missing
# any ONE of them leaves the run nondeterministic.
#
# Expected: with all three seeded the tuples are identical. With only two of them
#           seeded they are not, which is the trap: partial seeding looks like it
#           worked until the one unseeded library changes something that matters.
# ===============================================
def seeded_experiment(seed=0):
    torch.manual_seed(seed)
    # <-- seed numpy and the standard library here too
    weights = torch.randn(3)
    noise = np.random.randn(3)
    pick = random.choice(["a", "b", "c"])
    return round(weights.sum().item(), 4), round(float(noise.sum()), 4), pick

a = seeded_experiment(0)
b = seeded_experiment(0)
print("run 1:", a)
print("run 2:", b)
print("identical:", a == b)

## Part 2. Say what you ran it on

A seed is not enough. The same seed on a different library version can give
a different answer, so the versions are part of the result.

In [ ]:
# GIVEN. An environment record you can paste into a report.
def environment_record():
    import sklearn, transformers
    return {
        "python": sys.version.split()[0],
        "platform": f"{platform.system()} {platform.machine()}",
        "numpy": np.__version__,
        "torch": torch.__version__,
        "scikit-learn": sklearn.__version__,
        "transformers": transformers.__version__,
    }

for k, v in environment_record().items():
    print(f"   {k:<14} {v}")
print()
print("This project pins all of it in pyproject.toml and uv.lock, so anyone")
print("running `uv sync` gets these exact versions. Include the record anyway:")
print("a reader of your report should not have to clone your repository.")

In [ ]:
# ================== YOUR TURN 2 ==================
# Write the reproducibility checklist for YOUR project. Fill in each
# line honestly. Anything you cannot answer is a job to do before you
# submit.
#
# Expected: most projects fail at least one line the first time. The commonest
#           misses are an unseeded data split and results reported from a single
#           run with no sense of how much they move between seeds.
# ===============================================
CHECKLIST = {
    "every random source is seeded":            False,   # <-- edit these
    "the data split is fixed and saved":        False,
    "library versions are recorded":            True,
    "one command reproduces the main result":   False,
    "results are averaged over 3+ seeds":       False,
}

for item, done in CHECKLIST.items():
    print(f"   [{'x' if done else ' '}] {item}")

remaining = [k for k, v in CHECKLIST.items() if not v]
print(f"\n{len(remaining)} thing(s) to fix before you submit:")
for r in remaining:
    print("   -", r)

## Part 3. How much does the seed actually matter?

If your headline number moves by more than the difference you are claiming,
you have not measured anything yet.

In [ ]:
# GIVEN. The same tiny model trained under five seeds.
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=300, n_features=20, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
Xtr_t = torch.tensor(Xtr, dtype=torch.float32); ytr_t = torch.tensor(ytr)
Xte_t = torch.tensor(Xte, dtype=torch.float32); yte_t = torch.tensor(yte)

scores = []
for seed in range(5):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(20, 8), nn.ReLU(), nn.Linear(8, 2))
    opt = torch.optim.Adam(net.parameters(), lr=0.05)
    for _ in range(60):
        opt.zero_grad()
        nn.functional.cross_entropy(net(Xtr_t), ytr_t).backward()
        opt.step()
    with torch.no_grad():
        scores.append((net(Xte_t).argmax(1) == yte_t).float().mean().item())

print("accuracy by seed:", [round(s, 3) for s in scores])
print(f"mean {np.mean(scores):.3f}, spread {max(scores) - min(scores):.3f}")
print()
print("Report the mean AND the spread. A 0.01 improvement is not a result if")
print("the seed alone moves the number by more than that.")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   All three lines are needed:
#       torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
#   Partial seeding is worse than none, because it looks reproducible in testing
#   and is not.
#
# YOUR TURN 2
#   No single right answer. The two lines most projects fail are the fixed data
#   split and averaging over seeds. Both are cheap to fix now and impossible to
#   fix after you have written the results section.
#
# PART 3, nothing to edit
#   Compare the seed-to-seed spread against the effect you want to claim. If the
#   spread is bigger, the honest report is "no measurable difference", and
#   saying so is a better project than overclaiming.